# Working with GDM models

ERAD's `AssetSystem` provides a class method for users working with a GDM `DistributionSystem`. Distribution system components are automatically mapped to the asset models and added to the `AssetSystem`. A hazard simulation can then be set up using the steps listed above. We start by loading a GDM model using the `gdmloader` package. This package can be installed using the command

```shell
pip install gdmloader
```

In [ ]:
from IPython.display import display, HTML
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"

from gdm.distribution import DistributionSystem
from gdmloader.constants import GCS_CASE_SOURCE
from gdmloader.source import SystemLoader

gdm_loader = SystemLoader()
gdm_loader.add_source(GCS_CASE_SOURCE)

distribution_system: DistributionSystem = gdm_loader.load_dataset(
    source_name=GCS_CASE_SOURCE.name,
    system_type=DistributionSystem,
    dataset_name="p1rhs7_1247",
)
distribution_system.name = "p1rhs7_1247"
distribution_system.info()

/opt/homebrew/Caskroom/miniconda/base/envs/erad/lib/python3.12/site-packages/kaleido/__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




WARNING	Task(Task-2) google.auth.compute_engine._metadata:_metadata.py:ping()- Compute Engine Metadata server unavailable on attempt 1 of 3. Reason: [Errno 65] No route to host


WARNING	Task(Task-2) google.auth.compute_engine._metadata:_metadata.py:ping()- Compute Engine Metadata server unavailable on attempt 2 of 3. Reason: [Errno 64] Host is down


WARNING	Task(Task-2) google.auth.compute_engine._metadata:_metadata.py:ping()- Compute Engine Metadata server unavailable on attempt 3 of 3. Reason: [Errno 64] Host is down


WARNING	Task(Task-2) google.auth._default:_default.py:_get_gce_credentials()- Authentication failed using Compute Engine authentication due to unavailable metadata server.


...

WARNING	Task(Task-2) google.auth.compute_engine._metadata:_metadata.py:get()- Compute Engine Metadata server unavailable on attempt 2 of 5. Reason: HTTPConnectionPool(host='metadata.google.internal', port=80): Max retries exceeded with url: /computeMetadata/v1/instance/service-accounts/default/?recursive=true (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x16a286180>: Failed to resolve 'metadata.google.internal' ([Errno 8] nodename nor servname provided, or not known)"))


WARNING	Task(Task-2) google.auth.compute_engine._metadata:_metadata.py:get()- Compute Engine Metadata server unavailable on attempt 3 of 5. Reason: HTTPConnectionPool(host='metadata.google.internal', port=80): Max retries exceeded with url: /computeMetadata/v1/instance/service-accounts/default/?recursive=true (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x16a286ba0>: Failed to resolve 'metadata.google.internal' ([Errno 8] nodename nor servname provided, or not known)"))


WARNING	Task(Task-2) google.auth.compute_engine._metadata:_metadata.py:get()- Compute Engine Metadata server unavailable on attempt 4 of 5. Reason: HTTPConnectionPool(host='metadata.google.internal', port=80): Max retries exceeded with url: /computeMetadata/v1/instance/service-accounts/default/?recursive=true (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x16a287200>: Failed to resolve 'metadata.google.internal' ([Errno 8] nodename nor servname provided, or not known)"))


WARNING	Task(Task-2) google.auth.compute_engine._metadata:_metadata.py:get()- Compute Engine Metadata server unavailable on attempt 5 of 5. Reason: HTTPConnectionPool(host='metadata.google.internal', port=80): Max retries exceeded with url: /computeMetadata/v1/instance/service-accounts/default/?recursive=true (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x16a287710>: Failed to resolve 'metadata.google.internal' ([Errno 8] nodename nor servname provided, or not known)"))


ValidationError: 1 validation error for MatrixImpedanceBranch
  Value error, Length of matrix mat=<Quantity([[1.76096667 0.39596667]
 [0.39596667 1.76096667]], 'ohm / kilometer')> did not match number of phases self.phases=[<Phase.S2: 'S2'>, <Phase.S1: 'S1'>, <Phase.N: 'N'>] [type=value_error, input_value={'uuid': '42a039ae-76dd-4...tity(115.0, 'ampere')>)}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

Next, we built the asset system from the gdm `DistributionSystem` using the `from_gdm` method.

In [2]:
from erad.systems import AssetSystem

asset_system = AssetSystem.from_gdm(distribution_system)
asset_system.info()

for a in asset_system.iter_all_components():
    a.pprint()
    break

NameError: name 'distribution_system' is not defined

### Plotting an AssetSystem

In [3]:
fig = asset_system.plot(show=False)
fig.show()

NameError: name 'asset_system' is not defined

### Building a HazardModel

In this section, we built a hazard model and apply the model the asset system.

In [4]:
from datetime import datetime

from shapely.geometry import Polygon
from gdm.quantities import Distance

from erad.models.hazard import FloodModelArea, FloodModel
from erad.systems import HazardSystem
from erad.quantities import Speed

flood_area = FloodModelArea(
    affected_area=Polygon(
        [
            (-122.38, 38.70),
            (-122.35, 38.68),
            (-122.343, 38.69),
            (-122.37, 38.7035),
        ]
    ),
    water_velocity=Speed(0, "meter/second"),
    water_elevation=Distance(160, "meter"),
)

flood = FloodModel(
    name="flood 1",
    timestamp=datetime.now(),
    affected_areas=[flood_area],
)

user_defined_flood_event = HazardSystem(auto_add_composed_components=True)
user_defined_flood_event.add_component(flood)

### Overlaying the HazardModel

We can overlay the hazard model on the same plot using the `add_trace` method. The `show` method can be used to render the image again.

In [5]:
polygon = flood.affected_areas[0].affected_area
lon, lat = polygon.exterior.xy  # returns x and y sequences

fig.add_trace(
    go.Scattermap(
        fill="toself",
        lon=lon.tolist(),
        lat=lat.tolist(),
        marker={"size": 10, "color": flood.affected_areas[0].water_velocity.magnitude},
    )
)

fig.show()

NameError: name 'fig' is not defined

Finally, we can run the actual simulation using the `HazardSimulator` class from `erad.runner`.

In [6]:
from erad.runner import HazardSimulator

user_defined_flood_event.info()

hazard_scenario = HazardSimulator(asset_system=asset_system)
hazard_scenario.run(hazard_system=user_defined_flood_event)

System                                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Property                         ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ System name                      │       │
│ Data format version              │       │
│ Components attached              │     2 │
│ Time Series attached             │     0 │
│ Supplemental Attributes attached │     0 │
│ Description                      │       │
└──────────────────────────────────┴───────┘

Component Information           
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Type                 ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ FloodModel           │     1 │
│ FloodModelArea       │     1 │
└──────────────────────┴───────┘

NameError: name 'asset_system' is not defined

Once the simulation is complete, we can visualize the results by plotting the `survival_prob` from the updated gdf dataframe.

In [7]:
fig = asset_system.plot(show=False)
fig.show()

NameError: name 'asset_system' is not defined